In [2]:
import numpy as np
import h5py
import cv2 as cv
from pathlib import Path
from bin_picking.common.helper import load_dict_from_json


path = Path.cwd() / 'data' / 'image_processing'
rgb = cv.imread(str(path / 'color_image.png'))

In [ ]:
rgb_gray = cv.cvtColor(rgb, cv.COLOR_BGR2GRAY)

In [ ]:
ret, threshold = cv.threshold(rgb_gray, 130, 255, cv.THRESH_BINARY)

In [ ]:
res = cv.Canny(rgb_gray, 30, 120)

In [ ]:
dest = cv.cornerHarris(rgb_gray.astype(np.float32), 17, 21, 0.01)
img = rgb.copy()
img[dest > 0.01 * dest.max()] = (255,0,0)

In [ ]:
# cv.imshow('gray', rgb_gray)
# cv.imshow('rgb', rgb)
# cv.imshow('threshold', threshold)
cv.imshow('canny', res)
cv.waitKey(0)

In [152]:
contours, _ = cv.findContours(res, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
poly = np.zeros_like(rgb_gray)
# contour_image = cv.drawContours(np.zeros_like(rgb), contours, -1, (0,0,255), 2)
# image_contours = cv.drawContours(rgb.copy(), contours, -1, (255,0,0), 1)

for cnt in contours:
    epsilon = 0.1 * cv.arcLength(cnt, False)
    approx= cv.approxPolyDP(cnt, epsilon, True)
    if len(approx) == 4:
        cv.drawContours(poly, [approx], -1, 255, -1)

In [153]:
cv.imshow('processed', poly)
cv.waitKey(0)

-1

In [5]:
import open3d as o3
from scipy.fft import fft2, ifft2, fftfreq
f = h5py.File(str(path / 'data_set.hdf5'), 'r')
data_set = f['depth'][:]
f.close()
# data_set = cv.resize(data_set, (640, 480), interpolation=cv.INTER_NEAREST)
depth_info = load_dict_from_json(path / 'depth_camera_info.json')
intrinsic = depth_info['intrinsic']
cx, cy = intrinsic[2], intrinsic[5]
fx, fy = intrinsic[0], intrinsic[4]
depth = data_set
h,w = depth_info['resolution']
c_coords, v_coords = np.meshgrid(np.arange(w), np.arange(h))

depth_filtered = cv.bilateralFilter(depth.astype(np.float32),5, sigmaColor=1, sigmaSpace=5)


# F = fft2(depth)

# rows, cols = depth.shape
# crow, ccol = rows // 2, cols // 2

# Y, X = np.ogrid[:rows, :cols]
# dist = np.sqrt((Y - crow)**2 + (X - ccol)**2)

# sigma = 5
# gaussian = np.exp(-dist**2 / (2* sigma**2))

# F_shifted = np.fft.fftshift(F)
# F_filtered = F_shifted * gaussian
# depth_filtered = np.abs(ifft2(np.fft.ifftshift(F_filtered)))


X = (c_coords - cx) / fx * depth
Y = (v_coords - cy) / fy * depth
Z = depth.flatten()

X = X.flatten()
Y = Y.flatten()

xyz = np.column_stack([X,Y,Z])
mask = (depth_filtered < 3000) & (depth_filtered > 100)
xyz = xyz[mask.flatten()]

# mask = (poly > 0)
# print(data_set.shape)
# print(mask.shape)

pcd = o3.geometry.PointCloud(points=o3.utility.Vector3dVector(xyz))
# pcd.paint_uniform_color([0.5,0.5,0.5])
colors = np.asarray(pcd.colors)
o3.visualization.draw_geometries([pcd])